# Notebook 03 — Teach SmolLM2 maths with LoRA

Train a small adapter on top of the original `HuggingFaceTB/SmolLM2-360M`. The original model weights stay frozen; the adapter learns from the worked GSM8K solutions and their labelled final answers.

**Kaggle setup:** enable Internet, select a T4 GPU, and attach the **original saved Notebook 01 output** using Add Input. One GPU is used even if Kaggle supplies two T4s. You do not need to attach the pilot output for training; retain it for the later comparison.

This notebook audits tokenization, runs CPU contract tests, runs a two-update GPU smoke check, and then starts a fresh three-epoch training run. Checkpoints are selected using **validation response-token loss**, with no test-score selection. The saved adapter is not a standalone full model. Lower validation loss does not by itself prove higher generated maths accuracy; paired evaluation comes next.


## 1. Fetch the shared code

The launcher records the exact GitHub commit. Training logic and settings live in the repository. Keep the original data bundle fixed throughout the experiment.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

REPOSITORY = "https://github.com/sahan24854/Fine-Tuning-LLM-for-math.git"
PROJECT_REF = os.environ.get("MATH_PROJECT_REF", "main")
if PROJECT_REF.startswith("-"):
    raise ValueError("Use a branch, tag, or full commit SHA as MATH_PROJECT_REF")
PROJECT = Path(tempfile.mkdtemp(prefix="math-project-"))
subprocess.run(["git", "init", "--quiet", str(PROJECT)], check=True)
subprocess.run(["git", "-C", str(PROJECT), "remote", "add", "origin", REPOSITORY], check=True)
subprocess.run(["git", "-C", str(PROJECT), "fetch", "--depth", "1", "origin", PROJECT_REF], check=True)
subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", "--quiet", "FETCH_HEAD"], check=True)
CODE_COMMIT = subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
print("Repository:", REPOSITORY)
print("Code commit:", CODE_COMMIT)
print("Project checkout:", PROJECT)

Repository: https://github.com/sahan24854/Fine-Tuning-LLM-for-math.git
Code commit: 25f438582ec7f2d7de6affd7f574faad5714ffeb
Project checkout: /tmp/math-project-ofoc4s8w


From https://github.com/sahan24854/Fine-Tuning-LLM-for-math
 * branch            main       -> FETCH_HEAD
 * [new branch]      main       -> origin/main


## 2. Verify the original bundle and inspect the settings

The first run uses rank 16 LoRA on attention `q_proj`, `k_proj`, `v_proj`, and `o_proj`, learning rate `2e-4`, and three epochs. A micro-batch of four examples accumulated over eight micro-batches gives an effective batch of 32, except the last partial update of an epoch.

Training is ordinary LoRA with an FP16 frozen base and FP32 trainable adapters. No quantization or chat template is introduced. The supplied bundle records the original model revision and train/validation split.


In [2]:
import hashlib
import shutil

if not shutil.which("nvidia-smi"):
    raise RuntimeError("Enable a GPU accelerator in Kaggle settings before continuing.")
_ = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], check=True)
INPUT_ROOT = Path(os.environ.get("MATH_BUNDLE_PATH", "/kaggle/input"))
BUNDLE = Path(subprocess.check_output([sys.executable, str(PROJECT / "scripts/find_bundle.py"), str(INPUT_ROOT)], text=True).strip())
_ = subprocess.run([sys.executable, str(PROJECT / "scripts/verify_bundle.py"), str(BUNDLE)], check=True)
CONFIG_PATH = PROJECT / "configs/lora_training.json"
CONFIG = json.loads(CONFIG_PATH.read_text())
SOURCE_MANIFEST = json.loads((BUNDLE / "manifest.json").read_text())
BUNDLE_HASH = hashlib.sha256((BUNDLE / "manifest.json").read_bytes()).hexdigest()
CONFIG_HASH = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()
RUN_ROOT = Path("/kaggle/working") / f"lora-v1-{BUNDLE_HASH[:12]}-{CONFIG_HASH[:8]}-{CODE_COMMIT[:8]}"
SMOKE_OUTPUT, TRAIN_OUTPUT = RUN_ROOT / "smoke", RUN_ROOT / "full"
print("Bundle:", BUNDLE)
print("Pinned model:", SOURCE_MANIFEST["model"])
print("Original data counts:", SOURCE_MANIFEST["counts"])
print("Training settings:", json.dumps(CONFIG, indent=2))
print("Output:", RUN_ROOT)


Tesla T4, 15360 MiB, 580.178.04
Tesla T4, 15360 MiB, 580.178.04
Verified 17 files; code commit 6056c2f6cd6e096faf544ff026ba230f32639f9a
Bundle: /kaggle/input/notebooks/sahan24854/01-data-and-protocol/data-protocol-20261019T082909Z-1cb02106
Pinned model: {'id': 'HuggingFaceTB/SmolLM2-360M', 'revision': 'f8027fd0eaeea54caa13c31d31b9fdc459c38b49', 'revision_policy': 'Resolve once in notebook 01; every subsequent notebook must use config.resolved.json.'}
Original data counts: {'train': 6726, 'validation': 747, 'gsm8k_test': 1319, 'svamp_test': 1000, 'arithmetic_test': 500}
Training settings: {
  "training_id": "smollm2-360m-gsm8k-response-lora-1",
  "seed": 42,
  "epochs": 3,
  "learning_rate": 0.0002,
  "micro_batch_size": 4,
  "gradient_accumulation_steps": 8,
  "eval_batch_size": 4,
  "weight_decay": 0.01,
  "warmup_ratio": 0.03,
  "max_grad_norm": 1.0,
  "max_sequence_tokens": 1024,
  "checkpoint_every_updates": 100,
  "gradient_checkpointing": true,
  "lora": {
    "r": 16,
    "lora_

## 3. Prepare the training environment

This adds pinned PEFT and Accelerate packages to the known isolated PyTorch setup. The Kaggle kernel packages are not replaced. The shared setup avoids Kaggle's missing `ensurepip` problem.

The existing `wrapt` startup warning and `torch_dtype` deprecation warning may appear. CUDA execution and the following contract tests must pass before training.


In [3]:
ENVIRONMENT = Path(tempfile.mkdtemp(prefix="math-training-env-"))
_ = subprocess.run([sys.executable, str(PROJECT / "scripts/setup_environment.py"),
                    "--environment", str(ENVIRONMENT), "--profile", "training"], check=True)
PYTHON = ENVIRONMENT / "bin" / "python"
TRAIN_ENV = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
probe = """
import torch, peft, accelerate
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable: check Kaggle GPU settings.')
x = torch.ones((32, 32), device='cuda', dtype=torch.float16)
assert (x @ x).sum().item() == 32768
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', torch.cuda.get_device_name(0))
print('PEFT:', peft.__version__, 'Accelerate:', accelerate.__version__)
print('Visible GPUs for training:', torch.cuda.device_count())
"""
_ = subprocess.run([str(PYTHON), "-I", "-c", probe], check=True, env=TRAIN_ENV)


Creating isolated environment with Python 3.13.15


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


Processing /tmp/math-pip-bootstrap-q5v_e1_u/pip-25.3-py3-none-any.whl


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 120.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 102.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 113.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 186.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 21.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 2.0 MB/s  0:00:00



...

Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


No broken requirements found.
Environment ready: /tmp/math-training-env-ua6iezzw/bin/python


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


PyTorch: 2.7.1+cu118 CUDA: 11.8 GPU: Tesla T4
PEFT: 0.17.1 Accelerate: 1.10.1
Visible GPUs for training: 1


## 4. Verify the training contracts on CPU

Tiny local models check prompt/padding masking, EOS supervision, exact response-token loss, frozen base weights, validation-based selection, adapter reload, checkpoint recovery, and refusal to mix changed experiments. These tests do not download SmolLM2 or claim a maths improvement.


In [4]:
_ = subprocess.run([str(PYTHON), "-m", "unittest", "discover", "-s", "tests", "-p", "test_training.py", "-b", "-v"],
                    cwd=PROJECT, check=True, env=TRAIN_ENV)


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
test_cli_smoke_and_audit_use_only_training_and_validation (test_training.TrainingTests.test_cli_smoke_and_audit_use_only_training_and_validation) ... ok
test_loss_masks_prompt_and_padding_and_accumulates_by_response_tokens (test_training.TrainingTests.test_loss_masks_prompt_and_padding_and_accumulates_by_response_tokens) ... ok
test_real_checkpoint_resume_restores_training_and_matches_uninterrupted_adapter (test_training.TrainingTests.test_real_checkpoint_resume_restores_training_and_matches_uninterrupted_adapter) ... ok
test_real_training_freezes_base_selects_validation_and_reloads_adapter (test_training.TrainingTests.test_real_training_freezes_base_selects_validation_and_reloads_adapter) ... ok
test_segmented_encoding_eos_and_padding_never_supervise_prompt (test_training.TrainingTests.test_segmented_encoding_eos_and_padding_never_supervise_prompt) ... ok
test_validation_loss_is_token

## 5. Audit training and validation tokenization before loading weights

Each example is encoded as separately tokenized prompt + response + EOS. Prompt labels and padding are `-100`, so the loss trains the solution and final answer. Prompt text remains visible as context. There is no packing or truncation.

Every training token count must match Notebook 01's saved audit. Validation solutions are teacher-forced in the same zero-shot text format used for training. This is different from generating answers: it measures how well the model predicts the supplied correct solution tokens.

All original training/validation rows are used in the full run. If a sequence exceeds 1024 tokens, stop and inspect it rather than silently dropping or shortening it.


In [5]:
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/train_lora.py"),
                    "--bundle", str(BUNDLE), "--config", str(CONFIG_PATH), "--stage", "full",
                    "--output", str(RUN_ROOT / "audit"), "--audit-only"],
                    check=True, cwd=PROJECT, env=TRAIN_ENV)
AUDIT = json.loads((RUN_ROOT / "audit/data_audit.json").read_text())
print({key: AUDIT[key] for key in ("train_rows", "validation_rows", "max_train_tokens", "max_validation_tokens", "effective_batch_size", "planned_optimizer_updates")})


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


{
  "stage": "full",
  "train_rows": 6726,
  "validation_rows": 747,
  "train_ids": [
    "gsm8k/train/06582",
    "gsm8k/train/02732",
    "gsm8k/train/03855",
    "gsm8k/train/03607",
    "gsm8k/train/01795",
    "gsm8k/train/00526",
    "gsm8k/train/00726",
    "gsm8k/train/00822",
    "gsm8k/train/00024",
    "gsm8k/train/07469",
    "gsm8k/train/02298",
    "gsm8k/train/01182",
    "gsm8k/train/05818",
    "gsm8k/train/07132",
    "gsm8k/train/01080",
    "gsm8k/train/01958",
    "gsm8k/train/04786",
    "gsm8k/train/05484",
    "gsm8k/train/06285",
    "gsm8k/train/07436",
    "gsm8k/train/01231",
    "gsm8k/train/01665",
    "gsm8k/train/05706",
    "gsm8k/train/01738",
    "gsm8k/train/03806",
    "gsm8k/train/01888",
    "gsm8k/train/05160",
    "gsm8k/train/00426",
    "gsm8k/train/00781",
    "gsm8k/train/00886",
    "gsm8k/train/05189",
    "gsm8k/train/06152",
    "gsm8k/train/01532",
    "gsm8k/train/07386",
    "gsm8k/train/01524",
    "gsm8k/train/02184",
    "gsm8k/tra

## 6. Run two GPU optimizer updates as an engineering check

Use the first 64 training rows and 16 validation rows. This checks real model loading, LoRA gradients, finite losses/parameters, checkpoint saving, and an actual adapter change. It is not a useful trained model or an accuracy benchmark.

The full run starts again from the original model with a fresh adapter. It does not continue from these smoke weights. If this cell fails, inspect its error before running the full-training cell.


In [6]:
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/train_lora.py"),
                    "--bundle", str(BUNDLE), "--config", str(CONFIG_PATH), "--stage", "smoke",
                    "--output", str(SMOKE_OUTPUT)], check=True, cwd=PROJECT, env=TRAIN_ENV)
SMOKE = json.loads((SMOKE_OUTPUT / "summary.json").read_text())
if not SMOKE["health_passed"]:
    raise RuntimeError("Training smoke check failed; inspect the saved outputs.")
print("Smoke check:", json.dumps(SMOKE, indent=2))


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


Loading HuggingFaceTB/SmolLM2-360M at f8027fd0eaeea54caa13c31d31b9fdc459c38b49 on Tesla T4


`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded in 5.6s


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


trainable params: 3,276,800 || all params: 365,097,920 || trainable%: 0.8975
{'base_validation_loss': 0.9307494489481719, 'base_validation_response_tokens': 1653, 'base_validation_runtime': 0.7739, 'base_validation_samples_per_second': 20.674, 'base_validation_steps_per_second': 5.168}
{'loss': 0.9707, 'grad_norm': 0.13084985315799713, 'learning_rate': 0.0, 'epoch': 0.5}
{'loss': 0.9799, 'grad_norm': 0.1369597315788269, 'learning_rate': 0.0002, 'epoch': 1.0}
{'eval_loss': 0.9226939529631257, 'eval_response_tokens': 1653, 'eval_runtime': 0.5624, 'eval_samples_per_second': 28.451, 'eval_steps_per_second': 7.113, 'epoch': 1.0}
{'train_runtime': 10.5673, 'train_samples_per_second': 6.056, 'train_steps_per_second': 0.189, 'train_loss': 0.9753145277500153, 'epoch': 1.0}
{'selected_validation_loss': 0.9226939529631257, 'selected_validation_response_tokens': 1653, 'selected_validation_runtime': 0.5346, 'selected_validation_samples_per_second': 29.931, 'selected_validation_steps_per_second': 7.

## 7. Train the full adapter and select by validation loss

Train for three epochs. Evaluate and save every 100 optimizer updates and at epoch ends, keeping the best checkpoint plus recent recovery state. The selected adapter has the lowest mean loss over the supervised validation response tokens, including EOS. Short solutions are not given the same token weight as much longer ones.

The trainer records the initial base-model validation loss and all checkpoint losses. A decreasing training loss can accompany overfitting, so validation loss chooses the saved adapter. Neither loss establishes test accuracy or separates formatting gains from mathematical gains.

If interrupted, rerun this cell in the same session. It resumes the latest completed checkpoint, including optimizer/scheduler/random state. Changes to code, packages, configuration, bundle, or runtime are refused. Uncheckpointed updates must be repeated. Save the Kaggle outputs before the session ends; runtime identity checks may prevent continuing in a different environment.


In [7]:
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/train_lora.py"),
                    "--bundle", str(BUNDLE), "--config", str(CONFIG_PATH), "--stage", "full",
                    "--smoke-output", str(SMOKE_OUTPUT), "--output", str(TRAIN_OUTPUT)],
                    check=True, cwd=PROJECT, env=TRAIN_ENV)
SUMMARY = json.loads((TRAIN_OUTPUT / "summary.json").read_text())
print(json.dumps(SUMMARY, indent=2))


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


Loading HuggingFaceTB/SmolLM2-360M at f8027fd0eaeea54caa13c31d31b9fdc459c38b49 on Tesla T4


`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded in 1.3s


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


trainable params: 3,276,800 || all params: 365,097,920 || trainable%: 0.8975
{'base_validation_loss': 1.0395548987825187, 'base_validation_response_tokens': 71322, 'base_validation_runtime': 22.5691, 'base_validation_samples_per_second': 33.098, 'base_validation_steps_per_second': 8.286}
{'loss': 1.0088, 'grad_norm': 0.14392144978046417, 'learning_rate': 9.473684210526316e-05, 'epoch': 0.04756242568370987}
{'loss': 0.9823, 'grad_norm': 0.14326506853103638, 'learning_rate': 0.0002, 'epoch': 0.09512485136741974}
{'loss': 0.8608, 'grad_norm': 0.12811104953289032, 'learning_rate': 0.000196742671009772, 'epoch': 0.1426872770511296}
{'loss': 0.8129, 'grad_norm': 0.1466607004404068, 'learning_rate': 0.00019348534201954397, 'epoch': 0.1902497027348395}
{'loss': 0.8135, 'grad_norm': 0.13140924274921417, 'learning_rate': 0.00019022801302931596, 'epoch': 0.23781212841854935}
{'loss': 0.8168, 'grad_norm': 0.12975524365901947, 'learning_rate': 0.00018697068403908794, 'epoch': 0.2853745541022592}
{'

## 8. Review and save the adapter

`full/adapter` contains the selected adapter, not a merged copy of the 360M base model. Keep the complete ZIP, tokenizer, identity, base revision, config, validation history, and manifests. The archive also retains smoke results and recovery checkpoints.

For the next evaluation notebook, attach this saved output, the original Notebook 01 bundle, and the saved Notebook 02c pilot output. Both models must use the same 300 questions, prompts, stopping, and official scoring. Also inspect answer-format compliance and the documented arithmetic audit: printing `Final answer:` more reliably can improve the official score without an equivalent gain in mathematical ability.


In [8]:
from IPython.display import HTML, display
for output in (SMOKE_OUTPUT, TRAIN_OUTPUT):
    result = json.loads((output / "result_manifest.json").read_text())
    for name, expected in result["files_sha256"].items():
        if hashlib.sha256((output / name).read_bytes()).hexdigest() != expected:
            raise RuntimeError(f"Training result checksum mismatch: {name}")
display(HTML((TRAIN_OUTPUT / "report.html").read_text()))
ARCHIVE = Path(shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT))
print("Selected adapter:", TRAIN_OUTPUT / "adapter")
print("Training ZIP:", ARCHIVE)
print("Save Version with outputs. Next: paired evaluation against the saved baseline pilot.")


Update,Epoch,Response-token NLL
100,0.4756242568370987,0.766170
200,0.9512485136741974,0.738761
211,1.0,0.735921
300,1.4233055885850179,0.723382
400,1.8989298454221166,0.715532
422,2.0,0.713643
500,2.370986920332937,0.710608
600,2.8466111771700358,0.708155
633,3.0,0.707868


Selected adapter: /kaggle/working/lora-v1-b9d3419aa714-1100d832-25f43858/full/adapter
Training ZIP: /kaggle/working/lora-v1-b9d3419aa714-1100d832-25f43858.zip
Save Version with outputs. Next: paired evaluation against the saved baseline pilot.
